# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All entities (record sets, fields, columns, etc.) are referenced by their `@id` in accordance with the Croissant schema specification.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Cast to JSON so we can pretty print with keys
metadata = dataset.metadata.to_json()

print(f"Dataset Title: {metadata['name']}")
print('Description:')
print(metadata['description'])

## 2. Data Overview
Review available record sets, their field `@id`s, and which fields they contain. This helps you plan analysis using explicit schema `@id` referencing.

In [ ]:
# List all record sets and associated fields using their @id
print('Record sets, fields, and columns referenced by @id:')
for record_set in dataset.metadata.record_sets:
    print(f"\nRecord Set: {record_set['@id']} (name: {record_set.get('name', '')})")
    print('Fields:')
    for field in record_set['fields']:
        print(f"  - Field @id: {field['@id']} (name: {field.get('name', '')}, dataType: {field.get('dataType', '')})")
        if 'columns' in field:
            print('    Columns:')
            for col in field['columns']:
                print(f"     * Column @id: {col['@id']} (name: {col.get('name', '')})")

## 3. Data Extraction
Load data from a record set of interest into a DataFrame for analysis. All record set and field identifiers are referenced by their `@id`. For this dataset, you may find a primary record set such as `/protein` or `/main` (please verify actual `@id`, as schemas sometimes may use IDs like 'cr:RecordSet/0' or similar).

In [ ]:
# List all record set @ids for easy reference
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print('Available Record Set @ids:')
for r in record_set_ids:
    print(f' - {r}')

# For demonstration, pick the first record set @id (update to specific one as needed)
main_record_set_id = record_set_ids[0]

# Extract records for all record sets as DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"DataFrame loaded for record set {rs_id} ({len(records)} records)")
    else:
        print(f"No records found for record set {rs_id}")

if main_record_set_id in dataframes:
    print(f'Columns in {main_record_set_id}:')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

**Note:** For demonstration, we select a numeric field (e.g., coverage, abundance, or MW) and a grouping field (`description` or another string field). Adjust the field `@id`s based on your dataset's schema and exploration above.

In [ ]:
# Example: Suppose the numeric field @id is 'coverage' and the grouping field is 'description'. Adjust as needed based on exploration.

# Replace these with discovered @ids of numeric and group fields
numeric_field_id = None
group_field_id = None

if main_record_set_id in dataframes:
    # List all columns for interactive selection
    print('Available columns:', dataframes[main_record_set_id].columns.tolist())
    # Example: choose a common numeric field, try expected ones first
    for field_candidate in ['coverage', 'MW', 'abundance', 'PeptideCount', 'UniquePeptides', 'Abundance']:
        if field_candidate in dataframes[main_record_set_id].columns:
            numeric_field_id = field_candidate
            break
    for group_candidate in ['description', 'Accession', 'Group', 'ProteinGroup']:
        if group_candidate in dataframes[main_record_set_id].columns:
            group_field_id = group_candidate
            break

    if numeric_field_id is not None:
        threshold = dataframes[main_record_set_id][numeric_field_id].mean() if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][numeric_field_id]) else 0

        # Filter: greater than mean
        filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (first 5):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id is not None:
            grouped_df = (
                filtered_df.groupby(group_field_id)[numeric_field_id]
                .mean()
                .to_frame()
                .sort_values(numeric_field_id, ascending=False)
            )
            print(f"Grouped mean {numeric_field_id} by {group_field_id} (first 5 groups):")
            display(grouped_df.head())
        else:
            print("No suitable group_field_id found in columns.")
    else:
        print("No suitable numeric field found in columns.")
else:
    print(f"No DataFrame loaded for record set {main_record_set_id}")

## 5. Visualization
Visualize distributions or relationships between quantitative and qualitative fields using their schema `@id` (i.e., column names from the DataFrame).

Here, we provide a histogram for the selected numeric field and a bar plot for the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field distribution
if main_record_set_id in dataframes and numeric_field_id is not None:
    df = dataframes[main_record_set_id]
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.show()

    # Grouped means if group_field_id was found
    if group_field_id is not None:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped.values, y=grouped.index, orient='h')
        plt.xlabel(f'Mean {numeric_field_id}')
        plt.title(f'Top 10 {group_field_id} by mean {numeric_field_id}')
        plt.show()
else:
    print("Cannot visualize: No valid numeric field found or DataFrame not loaded.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset on protein abundance and modification analysis in human samples via the `mlcroissant` library. We've listed record sets, fields by `@id`, extracted records, filtered protein records, normalized numeric fields, and produced basic visualizations using schema-defined identifiers.

You can now extend the analysis to specific research questions—such as comparing groups, correlating attributes, or exporting clean data for downstream modeling—by referencing fields and records by their Croissant schema `@id` as illustrated.